# Dal notebook alla produzione

Il codice del capitolo [«Dal notebook alla produzione»](https://book.paithon.it/main/MLOps/dal-notebook-alla-produzione.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy scikit-learn scipy torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Dal notebook alla produzione

[Leggi la pagina](https://book.paithon.it/main/MLOps/dal-notebook-alla-produzione.html)


### Riproducibilità: i tre artefatti da versionare


In [ ]:
import random

import numpy as np
import torch


def fissa_seed(seed: int = 42) -> None:
    """Fissa le sorgenti di casualita' che il libro usa davvero."""
    random.seed(seed)
    np.random.seed(seed)      # sorgente "legacy" di NumPy
    torch.manual_seed(seed)   # pesi iniziali, dropout, DataLoader che mescola
    # i Generator moderni di NumPy ricevono il seme alla creazione:
    #   rng = np.random.default_rng(seed)
    # il DataLoader che mescola pesca dal seme globale; un generator
    # proprio lo isola dagli altri consumi (worker_init_fn serve solo per
    # i generatori che il loader non semina, come un np.random.Generator
    # creato a livello di modulo, che ogni worker riceverebbe identico):
    #   DataLoader(dati, shuffle=True,
    #              generator=torch.Generator().manual_seed(seed))

### Tracciare gli esperimenti


In [ ]:
import hashlib
import json


def hash_config(iperparametri: dict) -> str:
    """Impronta stabile della configurazione: stesso dict -> stesso hash."""
    # sort_keys rende irrilevante l'ordine con cui scriviamo le chiavi
    canonico = json.dumps(iperparametri, sort_keys=True).encode("utf-8")
    return hashlib.sha256(canonico).hexdigest()[:12]


def logga_run(registro: dict, iperparametri: dict, metriche: dict) -> str:
    """Registra un esperimento indicizzandolo per impronta di configurazione."""
    run_id = hash_config(iperparametri)
    registro[run_id] = {
        "iperparametri": iperparametri,
        "metriche": metriche,
    }
    return run_id


# --- uso: un registro in memoria, serializzabile in JSON ---
registro = {}

run_a = logga_run(
    registro,
    iperparametri={"lr": 1e-3, "batch_size": 64, "epoche": 5, "seed": 42},
    metriche={"val_accuracy": 0.973, "val_loss": 0.089},
)

# stessa configurazione, chiavi scritte in ordine diverso -> stesso identico id
run_b = hash_config({"seed": 42, "epoche": 5, "batch_size": 64, "lr": 1e-3})

print(run_a)            # e4d5dc4d91ef
print(run_a == run_b)   # True: l'impronta non dipende dall'ordine delle chiavi

## I dati contano quanto il programma

[Leggi la pagina](https://book.paithon.it/main/MLOps/dati-e-pipeline.html)


### Training–serving skew


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# --- addestramento ---
# una sola feature: l'importo di una transazione, in euro
X_train = rng.normal(loc=100.0, scale=20.0, size=10_000)

# statistiche "congelate" al momento dell'addestramento
mu, sigma = X_train.mean(), X_train.std()

# modellino lineare gia' addestrato sulla feature normalizzata z = (x - mu)/sigma
# lo score passa in una sigmoide -> probabilita' che la transazione sia una frode
w, b = 1.5, -0.2


def sigmoide(t):
    return 1.0 / (1.0 + np.exp(-t))


def predici_corretto(x):
    z = (x - mu) / sigma          # normalizza con le statistiche DEL TRAINING
    return sigmoide(w * z + b)


def predici_bacato(x_batch):
    # BUG: normalizza con media/std DEL BATCH corrente, non del training
    z = (x_batch - x_batch.mean()) / x_batch.std()
    return sigmoide(w * z + b)


# in produzione arriva un batch anomalo: importi molto piu' alti del solito
X_prod = rng.normal(loc=160.0, scale=20.0, size=32)

p_ok = predici_corretto(X_prod)
p_bug = predici_bacato(X_prod)

print("prob. media di frode (corretta):", round(float(p_ok.mean()), 3))
print("prob. media di frode (bacata):  ", round(float(p_bug.mean()), 3))
print("scarto massimo sulle predizioni:", round(float(np.abs(p_ok - p_bug).max()), 3))

### Validare i dati in ingresso


In [ ]:
from math import isnan

# schema: per ogni campo, il tipo atteso, l'intervallo ammesso e se e' obbligatorio
SCHEMA = {
    "eta":     {"tipo": int,   "min": 0,   "max": 120,   "obbligatorio": True},
    "importo": {"tipo": float, "min": 0.0, "max": 1e6,   "obbligatorio": True},
    "citta":   {"tipo": str,                             "obbligatorio": True},
}


def valida(record, schema):
    """Controlla un record (dict) contro lo schema; ritorna la lista degli errori."""
    errori = []
    for campo, regole in schema.items():
        # 1) valore assente o nullo
        if campo not in record or record[campo] is None:
            if regole.get("obbligatorio"):
                errori.append(f"{campo}: valore mancante")
            continue
        valore = record[campo]
        # 2) NaN, il "buco" numerico di NumPy/Pandas
        if isinstance(valore, float) and isnan(valore):
            errori.append(f"{campo}: NaN")
            continue
        # 3) tipo sbagliato
        if not isinstance(valore, regole["tipo"]):
            atteso = regole["tipo"].__name__
            errori.append(f"{campo}: tipo {type(valore).__name__}, atteso {atteso}")
            continue
        # 4) fuori dall'intervallo ammesso
        if "min" in regole and valore < regole["min"]:
            errori.append(f"{campo}: {valore} sotto il minimo {regole['min']}")
        if "max" in regole and valore > regole["max"]:
            errori.append(f"{campo}: {valore} oltre il massimo {regole['max']}")
    return errori


records = [
    {"eta": 34, "importo": 250.0, "citta": "Milano"},          # valido
    {"eta": 200, "importo": 90.0, "citta": "Roma"},            # eta fuori range
    {"eta": 41, "importo": float("nan"), "citta": "Napoli"},   # importo NaN
    {"eta": 29, "importo": 60.0},                              # citta mancante
    {"eta": "trenta", "importo": 15.0, "citta": "Torino"},     # eta di tipo sbagliato
]

for i, r in enumerate(records):
    errori = valida(r, SCHEMA)
    print(f"record {i}:", "OK" if not errori else " | ".join(errori))

### Collaudare il modello, non solo i dati


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)
positivi = ["ottimo", "buono", "delizioso", "gentile", "pulito", "rapido"]
negativi = ["pessimo", "cattivo", "freddo", "scortese", "sporco", "lento"]
cose = ["il servizio", "il piatto", "il cameriere", "il locale", "il conto", "il vino"]
citta = ["Milano", "Roma", "Napoli", "Torino", "Bari", "Genova"]

def frase(cosa, aggettivo, luogo):
    return f"a {luogo} {cosa} era {aggettivo}"

def campione(k):
    """Recensioni come quelle che arrivano davvero: nessuna con una negazione."""
    testi, etichette = [], []
    for _ in range(k):
        buona = rng.random() < 0.5
        agg = rng.choice(positivi if buona else negativi)
        testi.append(frase(rng.choice(cose), agg, rng.choice(citta)))
        etichette.append(int(buona))
    return testi, np.array(etichette)

X_tr, y_tr = campione(400)
X_te, y_te = campione(200)
vett = CountVectorizer().fit(X_tr)
modello = LogisticRegression().fit(vett.transform(X_tr), y_tr)
prevedi = lambda testi: modello.predict(vett.transform(testi))
print(f"accuratezza sul test: {(prevedi(X_te) == y_te).mean():.2f}")

# 1) funzionalità minima: la negazione rovescia il giudizio
negate = [frase(c, "non " + a, "Roma") for c in cose for a in positivi]
print(f"negazioni classificate negative: {(prevedi(negate) == 0).mean():.2f}")
# 2) invarianza: cambiare città non deve cambiare niente
base = [frase(c, a, "Milano") for c in cose for a in positivi + negativi]
altre = [frase(c, a, "Bari") for c in cose for a in positivi + negativi]
print(f"stessa risposta cambiando città: {(prevedi(base) == prevedi(altre)).mean():.2f}")
# 3) aspettativa direzionale: una lamentela in più deve abbassare il giudizio
p = lambda testi: modello.predict_proba(vett.transform(testi))[:, 1]
lodi = [frase(c, a, "Torino") for c in cose for a in positivi]
con_lamentela = [t + ", ma il conto era " + n for t, n in zip(lodi, negativi * 6)]
print(f"lodi che con una lamentela calano: {(p(con_lamentela) < p(lodi)).mean():.2f}")

## Servire un modello: dal file all'API

[Leggi la pagina](https://book.paithon.it/main/MLOps/deployment-e-serving.html)


### Il modello dietro un'API


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

import torch

# Caricamento UNA VOLTA all'avvio del servizio, non a ogni richiesta
modello = MiaRete()                        # la classe nn.Module usata in addestramento
modello.load_state_dict(torch.load("pesi.pt", map_location="cpu"))
modello.eval()                             # modalità inferenza: niente dropout, BatchNorm congelata

@torch.no_grad()                           # niente autograd: meno memoria, più veloce
def predici(richiesta: dict) -> dict:
    x = preprocessa(richiesta)             # dal JSON al tensore d'ingresso (batch di 1)
    logit = modello(x)                     # forward: logit grezzi, come in PyTorch
    prob = torch.softmax(logit, dim=1)     # logit -> probabilità
    return {
        "classe": int(prob.argmax(dim=1).item()),
        "confidenza": float(prob.max().item()),
    }
```


### Ottimizzare l'inferenza


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

import torch
import torch.nn as nn

# Pesi in int8, attivazioni quantizzate al volo (dynamic quantization).
# API storica, oggi in via di sostituzione: avvisa che si migri a torchao.
modello_int8 = torch.quantization.quantize_dynamic(
    modello, {nn.Linear}, dtype=torch.qint8,
)

esempio = torch.randn(1, 784)
programma = torch.export.export(modello, (esempio,))    # il grafo, catturato
torch.onnx.export(modello, (esempio,), "modello.onnx")  # lo stesso, in ONNX
```


## Sorvegliare un modello vivo

[Leggi la pagina](https://book.paithon.it/main/MLOps/monitoring-e-drift.html)


### Rilevare il drift in pratica


In [ ]:
import numpy as np
from scipy.stats import ks_2samp

rng = np.random.default_rng(0)
for n in (2000, 500):
    p = np.array([ks_2samp(rng.normal(0, 1, n), rng.normal(0.15, 1, n)).pvalue
                  for _ in range(2000)])
    print(f"n = {n}: rifiuti al 5% = {(p < 0.05).mean():.1%}, "
          f"p mediano = {np.median(p):.1e}")

In [ ]:
import numpy as np
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(0)

# Finestra di riferimento (il "passato" su cui il modello e' tarato)
# e finestra corrente (le ultime richieste arrivate in produzione).
n, d = 2000, 4
riferimento = rng.normal(0.0, 1.0, size=(n, d))
corrente = rng.normal(0.0, 1.0, size=(n, d))
corrente[:, 1] += 1.2   # drift iniettato solo sulla feature 1

def punteggio_drift(rif, cur):
    """AUC del detective: quanto e' facile distinguere le due finestre."""
    X = np.vstack([rif, cur])
    y = np.hstack([np.zeros(len(rif)), np.ones(len(cur))])
    detective = HistGradientBoostingClassifier(random_state=0)
    return cross_val_score(detective, X, y, cv=5, scoring="roc_auc").mean()

SOGLIA = 0.65  # AUC oltre la quale scatta l'allarme
auc = punteggio_drift(riferimento, corrente)
print(f"AUC detective = {auc:.3f}")

if auc > SOGLIA:
    print(f"ALLARME: drift rilevato (AUC {auc:.3f} > {SOGLIA})")
    # Localizziamo: un test di Kolmogorov-Smirnov per ogni feature.
    for j in range(d):
        stat, p = ks_2samp(riferimento[:, j], corrente[:, j])
        # si segnala sull'ampiezza, non sul p-value: a questa taglia di
        # finestra il p e' minuscolo anche su scostamenti che nessuno sente
        sospetta = "  <-- sospetta" if stat > 0.10 else ""
        print(f"  feature {j}: KS={stat:.3f}  p={p:.1e}{sospetta}")
else:
    print("Nessun drift rilevabile: il detective non distingue le finestre.")

# Dove il detective arriva e dove no: quaranta colonne, e lo stesso
# scostamento per colonna (0,15 deviazioni standard) prima su una colonna
# sola, poi su tutte e quaranta. Il KS si legge sulle colonne DERIVATE, non sul
# massimo delle quaranta: il massimo cresce con quante colonne guardi, quindi
# confrontarlo fra i due casi misurerebbe la selezione e non la deriva.
n_colonne = 40
rif40 = rng.normal(0.0, 1.0, size=(n, n_colonne))
for etichetta, derivate in [("su una colonna sola", [1]),
                            ("su tutte e quaranta", list(range(n_colonne)))]:
    cur40 = rng.normal(0.0, 1.0, size=(n, n_colonne))
    cur40[:, derivate] += 0.15
    ks = [ks_2samp(rif40[:, j], cur40[:, j]).statistic for j in range(n_colonne)]
    tipico = sorted(ks[j] for j in derivate)[len(derivate) // 2]
    print(f"deriva {etichetta:19}: AUC = {punteggio_drift(rif40, cur40):.3f}, "
          f"KS tipico delle derivate = {tipico:.3f}, "
          f"colonne oltre la soglia: {sum(v > 0.10 for v in ks)}")

### Quanta strada, e quanto bene ordina


In [ ]:
import numpy as np
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
n = 20000
riferimento = rng.normal(0, 1, n)                  # una colonna già riscalata: l'unità è la deviazione standard
# due derive diverse: tutti un po' più in là, oppure pochi molto lontano
tutti_poco = riferimento + 0.2
pochi_molto = riferimento.copy()
pochi_molto[: n // 50] += 10                       # due clienti su cento, dieci unità più in là
for nome, corrente in [("tutti di 0,2", tutti_poco), ("il 2% di 10", pochi_molto)]:
    print(f"{nome:13}: KS {ks_2samp(riferimento, corrente).statistic:.3f},"
          f" Wasserstein {wasserstein_distance(riferimento, corrente):.3f}")

# prestazioni, quando arrivano le etichette: punteggi dei buoni e dei cattivi pagatori
cattivo = rng.random(n) < 0.1
punteggio = rng.normal(0, 1, n) + 1.2 * cattivo
auc = roc_auc_score(cattivo, punteggio)
# il Gini del credito dalla curva CAP: quota di cattivi presi contro quota di clienti scartati
ordine = np.argsort(-punteggio)
presi = np.r_[0, np.cumsum(cattivo[ordine])] / cattivo.sum()
scartati = np.linspace(0, 1, n + 1)
area = np.sum((presi[1:] + presi[:-1]) / 2 * np.diff(scartati)) - 0.5   # sopra la diagonale
perfetta = 0.5 * (1 - cattivo.mean())                                      # il modello perfetto
print(f"AUC {auc:.4f}; 2 AUC - 1 = {2 * auc - 1:.4f}; Gini dalla curva CAP = {area / perfetta:.4f}")
print(f"KS fra i punteggi dei buoni e dei cattivi: "
      f"{ks_2samp(punteggio[cattivo], punteggio[~cattivo]).statistic:.3f}")

### Quando è cambiato: la somma che si accumula


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def cusum(X, k, h):
    """Somma cumulata di Page su molte serie insieme (una per riga):
    S_t = max(0, S_{t-1} + x_t - k), allarme al primo S_t > h.
    Restituisce, per ogni serie, il giorno dell'allarme contato da 1
    (-1 se non scatta)."""
    S = np.zeros(X.shape[0])
    allarme = np.full(X.shape[0], -1)
    for t in range(X.shape[1]):
        S = np.maximum(0.0, S + X[:, t] - k)
        allarme[(S > h) & (allarme < 0)] = t + 1
    return allarme

K = 0.25                              # metà del peggioramento da cogliere (0,5)
for h in (2, 4, 8):
    quiete = rng.normal(0, 1, size=(1000, 5000))      # niente è cambiato
    falsi = cusum(quiete, K, h)
    tra_un_falso_e_laltro = np.where(falsi < 0, 5000, falsi).mean()
    dopo = rng.normal(0.5, 1, size=(1000, 1000))      # la media è salita di 0,5
    ritardo = cusum(dopo, K, h)
    assert (ritardo >= 0).all()                       # dopo il cambio suona sempre
    print(f"soglia h = {h}: un falso allarme ogni {tra_un_falso_e_laltro:4.0f} giorni"
          f" in media; dopo il cambio suona in {ritardo.mean():4.1f} giorni")

## Misurare la generazione: TTFT, TPOT e goodput

[Leggi la pagina](https://book.paithon.it/main/MLOps/metriche-di-servizio.html)


### Misurare in venti righe


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

SLO_TTFT, SLO_TPOT = 0.500, 0.050   # obiettivi dichiarati: 500 ms e 50 ms
FINESTRA = 10.0                     # secondi di traffico osservato

def misura(nome, n, ttft_mediano, tpot_mediano, sigma=0.35):
    """Simula n richieste servite nella finestra e ne riassume le metriche."""
    ttft = rng.lognormal(np.log(ttft_mediano), sigma, n)  # code lunghe a destra
    tpot = rng.lognormal(np.log(tpot_mediano), sigma, n)
    ok = (ttft <= SLO_TTFT) & (tpot <= SLO_TPOT)          # rispetta ENTRAMBE
    sfora = (ttft > SLO_TTFT).mean()                      # guarda solo il TTFT
    p50, p95, p99 = np.percentile(ttft, [50, 95, 99]) * 1000
    print(f"{nome:<9}{n / FINESTRA:8.1f}{ok.sum() / FINESTRA:9.1f}{ok.mean():10.1%}"
          f"{sfora:9.1%}{ttft.mean() * 1000:9.0f}{p50:7.0f}{p95:7.0f}{p99:7.0f}")

print(f"{'config':<9}{'ric/s':>8}{'good/s':>9}{'conformi':>10}{'TTFT>SLO':>9}"
      f"{'TTFTmed':>9}{'p50':>7}{'p95':>7}{'p99':>7}")
misura("batch 16", 200, 0.28, 0.028)
misura("batch 64", 320, 0.43, 0.043)